# ZS601 2DGS Pro+ smoke run

This notebook is the Colab entrypoint for the clean `2dgs-zs601-mask-init` branch.

Goal for the first run:

1. Start a GPU runtime, preferably T4 or better under Colab Pro+.
2. Mount Google Drive so the ZS601 data zip can be read.
3. Clone the clean 2DGS-only GitHub branch.
4. Build the CUDA extensions in the current runtime.
5. Stage the ZS601 dataset under `/content` without modifying the original Drive files.
6. Run a 200-step COLMAP-sparse smoke test with mask-aware loss.
7. Save logs and smoke outputs to a new timestamped output directory.

Do not run the long formal experiment until this smoke notebook finishes cleanly.


In [ ]:
# 0. Runtime and GPU evidence
from pathlib import Path
import os, sys, json, time, shutil, zipfile, subprocess, textwrap

def sh(cmd, cwd=None, check=True):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")
    return result

print('Python:', sys.version)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
sh('nvidia-smi')
try:
    import torch
    print('Torch:', torch.__version__)
    print('Torch CUDA:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('Capability:', torch.cuda.get_device_capability(0))
except Exception as exc:
    print('Torch probe failed:', repr(exc))
    raise


In [ ]:
# 1. Mount Drive. If Colab asks for Google authorization, the user must approve it manually.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Run configuration
from pathlib import Path
import time, json, os

RUN_ID = 'zs601_2dgs_proplus_smoke_' + time.strftime('%Y%m%d_%H%M%S')
REPO_URL = 'https://github.com/VISjudy/ZS601_3DGS.git'
BRANCH = '2dgs-zs601-mask-init'
WORK_ROOT = Path('/content') / RUN_ID
REPO_ROOT = WORK_ROOT / 'repo'
SOURCE_ROOT = REPO_ROOT / '2d-gaussian-splattingWithMask'
RAW_UNZIP_ROOT = WORK_ROOT / 'raw_unzip'
STAGED_DATASET = WORK_ROOT / 'ZS601meetingroom_data_staged'
LOCAL_OUTPUT_ROOT = Path('/content/outputs') / RUN_ID
DRIVE_DATA_ZIP = Path('/content/drive/MyDrive/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/LCCDataset/zs601_output') / RUN_ID

for path in [WORK_ROOT, RAW_UNZIP_ROOT, LOCAL_OUTPUT_ROOT]:
    path.mkdir(parents=True, exist_ok=False)

run_spec = {
    'run_id': RUN_ID,
    'repo_url': REPO_URL,
    'branch': BRANCH,
    'source_root': str(SOURCE_ROOT),
    'drive_data_zip': str(DRIVE_DATA_ZIP),
    'staged_dataset': str(STAGED_DATASET),
    'local_output_root': str(LOCAL_OUTPUT_ROOT),
    'drive_output_root': str(DRIVE_OUTPUT_ROOT),
    'smoke_iterations': 200,
    'mask_valid_value': 'black',
}
(LOCAL_OUTPUT_ROOT / 'run_spec.json').write_text(json.dumps(run_spec, indent=2), encoding='utf-8')
print(json.dumps(run_spec, indent=2))


In [ ]:
# 3. Clone the clean 2DGS-only branch and verify source identity
sh(f'git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_ROOT}')
head = sh('git rev-parse HEAD', cwd=REPO_ROOT).stdout.strip()
print('Cloned HEAD:', head)
assert SOURCE_ROOT.exists(), SOURCE_ROOT
sh('git ls-tree --name-only HEAD', cwd=REPO_ROOT)


In [ ]:
# 4. Install dependencies and build CUDA extensions
# T4 compute capability is 7.5; this also works on many Colab GPUs, but can be adjusted if needed.
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
sh('python -m pip install -q --upgrade pip')
sh('python -m pip install -q plyfile tqdm numpy scipy pillow trimesh opencv-python scikit-image')
sh('python -m pip install -q submodules/diff-surfel-rasterization', cwd=SOURCE_ROOT)
sh('python -m pip install -q submodules/simple-knn', cwd=SOURCE_ROOT)
sh('python - <<"PY"\nimport diff_surfel_rasterization\nimport simple_knn\nprint("CUDA extension imports OK")\nPY', cwd=SOURCE_ROOT)


In [ ]:
# 5. Stage ZS601 dataset from Drive without modifying the original zip or Drive source files
assert DRIVE_DATA_ZIP.exists(), f'Missing data zip: {DRIVE_DATA_ZIP}'
print('Data zip size GB:', DRIVE_DATA_ZIP.stat().st_size / 1024**3)

with zipfile.ZipFile(DRIVE_DATA_ZIP) as zf:
    names = zf.namelist()[:20]
    print('Zip first entries:', names)
    zf.extractall(RAW_UNZIP_ROOT)

def find_dataset_root(base: Path) -> Path:
    candidates = []
    for sparse in base.rglob('sparse'):
        if (sparse / 'cameras.txt').exists() or (sparse / 'images.txt').exists() or (sparse / '0' / 'cameras.txt').exists() or (sparse / '0' / 'images.txt').exists():
            parent = sparse.parent
            if (parent / 'images').exists():
                candidates.append(parent)
    if not candidates:
        raise FileNotFoundError(f'Could not find COLMAP dataset root under {base}')
    candidates.sort(key=lambda p: len(str(p)))
    return candidates[0]

raw_dataset = find_dataset_root(RAW_UNZIP_ROOT)
print('Raw dataset root:', raw_dataset)
shutil.copytree(raw_dataset, STAGED_DATASET, dirs_exist_ok=False)

sparse = STAGED_DATASET / 'sparse'
sparse0 = sparse / '0'
if not sparse0.exists():
    sparse0.mkdir(parents=True, exist_ok=False)
    for name in ['cameras.txt', 'images.txt', 'points3D.txt', 'cameras.bin', 'images.bin', 'points3D.bin']:
        src = sparse / name
        if src.exists():
            shutil.copy2(src, sparse0 / name)

print('Staged dataset:', STAGED_DATASET)
print('sparse/0 files:', sorted(p.name for p in sparse0.iterdir()))
print('images exists:', (STAGED_DATASET / 'images').exists())
print('masks exists:', (STAGED_DATASET / 'masks').exists())


In [ ]:
# 6. Validate COLMAP image paths and mask matches before training
from pathlib import Path

def read_colmap_image_names(images_txt: Path):
    result = []
    lines = [ln.strip() for ln in images_txt.read_text(encoding='utf-8', errors='ignore').splitlines() if ln.strip() and not ln.startswith('#')]
    for i in range(0, len(lines), 2):
        parts = lines[i].split()
        if len(parts) >= 10:
            result.append(parts[9])
    return result

image_names = read_colmap_image_names(STAGED_DATASET / 'sparse' / '0' / 'images.txt')
print('COLMAP registered images:', len(image_names))
missing_images, missing_masks = [], []
for rel in image_names:
    img = STAGED_DATASET / 'images' / rel
    mask = STAGED_DATASET / 'masks' / Path(rel).with_suffix('.png')
    if not img.exists():
        missing_images.append(str(img))
    if not mask.exists():
        missing_masks.append(str(mask))
print('Missing image count:', len(missing_images))
print('Missing mask count:', len(missing_masks))
if missing_images[:5]: print('Missing image examples:', missing_images[:5])
if missing_masks[:5]: print('Missing mask examples:', missing_masks[:5])
assert not missing_images, 'Image path mismatch'
assert not missing_masks, 'Mask path mismatch'


In [ ]:
# 7. 200-step smoke training with mask-aware 2DGS loss
import subprocess, json, time

SMOKE_OUT = LOCAL_OUTPUT_ROOT / 'colmap_sparse_mask_smoke_200'
LOG_PATH = LOCAL_OUTPUT_ROOT / 'smoke_train.log'
cmd = [
    'python', 'train.py',
    '-s', str(STAGED_DATASET),
    '--images', 'images',
    '--masks', 'masks',
    '--mask_valid_value', 'black',
    '-m', str(SMOKE_OUT),
    '--eval',
    '--iterations', '200',
    '--sh_degree', '2',
    '--position_lr_init', '0.000016',
    '--position_lr_final', '0.00000016',
    '--position_lr_max_steps', '150000',
    '--scaling_lr', '0.0015',
    '--densification_interval', '100',
    '--densify_until_iter', '200',
    '--opacity_reset_interval', '150000',
    '--densify_grad_threshold', '0.0002',
    '--lambda_normal', '0.0',
    '--lambda_dist', '0.0',
    '--depth_ratio', '0',
    '--test_iterations', '200',
    '--save_iterations', '200',
    '--checkpoint_iterations', '200',
    '--quiet',
]
print('Smoke command:')
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd=SOURCE_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
LOG_PATH.write_text(proc.stdout, encoding='utf-8')
print(proc.stdout[-8000:])
print('Return code:', proc.returncode)
if proc.returncode != 0:
    failure = {
        'phase': 'smoke_training',
        'returncode': proc.returncode,
        'log_path': str(LOG_PATH),
        'tail': proc.stdout[-4000:],
    }
    (LOCAL_OUTPUT_ROOT / 'failure.json').write_text(json.dumps(failure, indent=2), encoding='utf-8')
    raise RuntimeError('Smoke training failed; see smoke_train.log')
print('Smoke output:', SMOKE_OUT)


In [ ]:
# 8. Verify smoke artifacts and copy them to a new Drive output folder
import json, shutil

expected = {
    'smoke_out_exists': SMOKE_OUT.exists(),
    'point_cloud_iter_200': (SMOKE_OUT / 'point_cloud' / 'iteration_200').exists(),
    'checkpoint_200': (SMOKE_OUT / 'chkpnt200.pth').exists(),
    'log_path': str(LOG_PATH),
    'local_output_root': str(LOCAL_OUTPUT_ROOT),
}
print(json.dumps(expected, indent=2))
assert expected['smoke_out_exists'], SMOKE_OUT
assert expected['point_cloud_iter_200'], SMOKE_OUT / 'point_cloud' / 'iteration_200'
assert expected['checkpoint_200'], SMOKE_OUT / 'chkpnt200.pth'

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
shutil.copy2(LOCAL_OUTPUT_ROOT / 'run_spec.json', DRIVE_OUTPUT_ROOT / 'run_spec.json')
shutil.copy2(LOG_PATH, DRIVE_OUTPUT_ROOT / 'smoke_train.log')
shutil.copytree(SMOKE_OUT, DRIVE_OUTPUT_ROOT / 'colmap_sparse_mask_smoke_200', dirs_exist_ok=False)
verification = dict(expected)
verification['drive_output_root'] = str(DRIVE_OUTPUT_ROOT)
verification['status'] = 'smoke_passed'
(DRIVE_OUTPUT_ROOT / 'verification.json').write_text(json.dumps(verification, indent=2), encoding='utf-8')
print('Saved Drive output:', DRIVE_OUTPUT_ROOT)
print((DRIVE_OUTPUT_ROOT / 'verification.json').read_text())


## Formal run gate

Only run a formal 150k experiment after the 200-step smoke passes and the saved `verification.json` is read back from Drive.

Recommended formal command keeps the same LR horizon as the smoke command:

```bash
python train.py -s <staged_dataset> --images images --masks masks --mask_valid_value black \
  -m /content/outputs/<new_formal_output> --eval --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --lambda_normal 0.05 --lambda_dist 0.0 --depth_ratio 0 \
  --test_iterations 50000 100000 150000 \
  --save_iterations 50000 100000 150000 \
  --checkpoint_iterations 50000 100000 150000 --quiet
```
